# 02 · Limpieza de datos — Adopción de IA en empresas (2023–2026)

Este cuaderno aplica sobre el archivo crudo las decisiones del **registro de decisiones** y guarda el resultado en `datos/derivados/ai_company_adoption_limpio.parquet`. Cada paso se verifica con `assert`, y la tabla final `pasos` documenta cuántas filas y columnas quedan después de cada uno.

El diccionario con el perfil y la decisión por variable está en `01_diccionario_ai.ipynb`.

Este dataset llega **sin nulos, sin filas duplicadas y sin valores imposibles**, así que la limpieza no descarta ni modifica ningún valor: valida la estructura, fija los tipos (categorías ordenadas) y guarda el archivo derivado.

## Registro de decisiones

| # | Qué | Qué se hizo | Evidencia | Por qué (alternativa descartada) | Pregunta |
|---|---|---|---|---|---|
| 1 | Unidad de análisis | Empresa-trimestre: se usan las 150.000 filas | 10.000 empresas con entre 10 y 16 filas cada una | Promediar por empresa o fijar un año descartaría información; se decidió analizar todas las filas | P1–P10 |
| 2 | `response_id` | Se conserva sin completar | Llega a 160.000 pero hay 150.000 filas (10.000 saltos); Kaggle muestra lo mismo | No se explica el motivo y no se pueden inventar los registros que faltan (alternativa descartada: completar los trimestres ausentes) | P1–P10 |
| 3 | 2026 completo | Se conserva | 4 trimestres con unas 9.400 filas cada uno, aunque el Q4 de 2026 no había terminado en oct-2026 | No se puede verificar a qué corresponden; se aclara en la ficha | P1–P10 |
| 4 | Valores imposibles | No se modifica ningún valor | 14 chequeos con 0 filas (porcentajes, `company_age`, `years_using_ai`, puestos contra `num_employees`) | No hay nada que corregir | P1–P10 |
| 5 | Pisos y topes | Se conservan como dato | `cost_reduction_percent` 15.887 filas en 0, `productivity_change_percent` 9.027 en 0, `revenue_growth_percent` 6.106 en −5 | No son faltantes ni errores comprobables (alternativa descartada: tratar los ceros como nulos) | P4, P7, P8 |
| 6 | Atípicos | Se conservan, sin recortar ni eliminar | `jobs_displaced`: mediana 10, máximo 2.417, nunca supera `num_employees`; la inversión total (por empleado × empleados) es como máximo el 25 % de los ingresos anuales | No hay evidencia de error; eliminar o recortar cambiaría los promedios (alternativa descartada: winsorizar) | P6 |
| 7 | `ai_adoption_stage` y `years_using_ai` | Se conservan | La etapa es la tasa de adopción en tramos y `years_using_ai` = 0 exactamente cuando la etapa es `none` | Dicen en parte lo mismo que `ai_adoption_rate`; se tiene en cuenta al analizar | P1–P3, P6 |
| 8 | Categóricas | Ordinales: `quarter`, `company_age_group`, `ai_adoption_stage`, `data_privacy_level`. El resto, categóricas sin orden | El diccionario las clasifica así; `company_size` figura como nominal y sus umbrales no están documentados | El orden importa para tablas y gráficos; no se inventa un orden entre Startup, SME y Enterprise | P1–P3 |
| 9 | Ponderación | No hay | El dataset no trae una variable de pesos | Todas las filas pesan igual | P1–P10 |

## 1. Lectura

El archivo se lee tal cual. Se desactivan los NA automáticos de pandas porque `none` es una **etapa de adopción** válida (`ai_adoption_stage`), no un faltante.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'proyecto_equipo' / 'datos').is_dir())
PROYECTO = RAIZ / 'proyecto_equipo'
RUTA = PROYECTO / 'datos' / 'crudos' / 'ai_company_adoption.csv'
SALIDA = PROYECTO / 'datos' / 'derivados' / 'ai_company_adoption_limpio.parquet'
SALIDA.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RUTA, keep_default_na=False, na_values=[''])
pasos = [('lectura del crudo', len(df), df.shape[1])]

## 2. Estructura (registros #1 a #3 y #9)

No se descarta nada: se verifica que el archivo tenga la estructura que se documentó en el diccionario.

In [2]:
assert len(df) == 150_000 and df.shape[1] == 43
assert df.isna().sum().sum() == 0                  # sin celdas nulas
assert not df.duplicated().any()                    # sin filas duplicadas
assert df['response_id'].is_unique

# Una empresa tiene entre 10 y 16 filas; panel no balanceado
assert df.groupby('company_id').size().between(10, 16).all()
assert df['company_id'].nunique() == 10_000

# Estos atributos no cambian dentro de una empresa
fijas = ['country', 'region', 'industry', 'company_size', 'company_founding_year',
         'num_employees', 'annual_revenue_usd_millions']
assert (df.groupby('company_id')[fijas].nunique() == 1).all().all()

pasos.append(('verificación de estructura (no se descarta nada)', len(df), df.shape[1]))

## 3. Valores imposibles y relaciones (registros #4, #5 y #7)

Un día no puede tener más de 100 % de nada, ninguna empresa puede desplazar más puestos que empleados, y la antigüedad tiene que ser coherente con el año de la fila. Si algún chequeo fallara, el cuaderno se frena antes de guardar.

In [3]:
pct = ['ai_adoption_rate', 'ai_budget_percentage', 'remote_work_percentage', 'task_automation_rate',
       'ai_failure_rate', 'productivity_change_percent', 'cost_reduction_percent']
for c in pct:
    assert df[c].between(0, 100).all(), c
assert (df['revenue_growth_percent'] >= -100).all()
assert (df['time_saved_per_week'] <= 168).all()

assert (df['company_age'] == df['survey_year'] - df['company_founding_year']).all()
assert (df['years_using_ai'] <= df['company_age']).all()
for c in ['jobs_displaced', 'jobs_created', 'reskilled_employees']:
    assert (df[c] <= df['num_employees']).all(), c

# La etapa de adopción es la tasa cortada en tramos; years_using_ai = 0 solo en la etapa none
tramos = df.groupby('ai_adoption_stage')['ai_adoption_rate'].agg(['min', 'max'])
assert tramos.loc['none', 'max'] <= 10.01
assert tramos.loc['pilot', 'min'] >= 9.99 and tramos.loc['pilot', 'max'] <= 35.01
assert tramos.loc['partial', 'min'] >= 34.99 and tramos.loc['partial', 'max'] <= 70.01
assert tramos.loc['full', 'min'] >= 69.99
assert ((df['years_using_ai'] == 0) == (df['ai_adoption_stage'] == 'none')).all()

# Pisos y topes que se conservan como dato (registro #5)
print('cost_reduction_percent en 0:     ', int((df['cost_reduction_percent'] == 0).sum()))
print('productivity_change_percent en 0:', int((df['productivity_change_percent'] == 0).sum()))
print('revenue_growth_percent en −5:    ', int((df['revenue_growth_percent'] == -5).sum()))

cost_reduction_percent en 0:      15887
productivity_change_percent en 0: 9027
revenue_growth_percent en −5:     6106


Se conservan los pisos y topes y los atípicos tal cual vienen (registros #5 y #6): no hay evidencia de que sean errores, y recortarlos cambiaría los promedios de las preguntas.

In [4]:
pasos.append(('validación de rangos y relaciones (no se modifica nada)', len(df), df.shape[1]))

## 4. Tipos categóricos (registro #8)

Las categorías con orden natural se declaran **ordinales**. Las demás pasan a `category` sin orden: ocupan menos memoria y evitan comparar texto por error.

In [5]:
ordenes = {
    'quarter':           ['Q1', 'Q2', 'Q3', 'Q4'],
    'company_age_group': ['0-5 years', '6-15 years', '16-30 years', '30+ years'],
    'ai_adoption_stage': ['none', 'pilot', 'partial', 'full'],
    'data_privacy_level': ['Low', 'Medium', 'High'],
}
for c, orden in ordenes.items():
    assert set(df[c].unique()) == set(orden), f'{c}: hay categorías fuera de la lista'
    df[c] = pd.Categorical(df[c], categories=orden, ordered=True)

nominales = ['country', 'region', 'industry', 'company_size', 'ai_primary_tool', 'ai_use_case',
             'ai_ethics_committee', 'survey_source', 'data_collection_method']
for c in nominales:
    df[c] = df[c].astype('category')

assert df.isna().sum().sum() == 0                   # ninguna categoría quedó fuera de la lista
print(f'memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
pasos.append(('categorías ordinales y nominales', len(df), df.shape[1]))

memoria: 39.5 MB


## 5. Verificación final y guardado

In [6]:
assert len(df) == 150_000 and df.shape[1] == 43
df.to_parquet(SALIDA, index=False)

leido = pd.read_parquet(SALIDA)
assert leido.shape == df.shape
for c, orden in ordenes.items():
    assert leido[c].cat.ordered and list(leido[c].cat.categories) == orden
num = df.select_dtypes('number').columns
assert np.allclose(leido[num].sum(), df[num].sum())
pasos.append(('guardado en derivados', len(leido), leido.shape[1]))

pd.DataFrame(pasos, columns=['paso', 'filas', 'columnas'])

,paso,filas,columnas
0,lectura del crudo,150000,43
1,verificación de estructura (no se descarta nada),150000,43
2,validación de rangos y relaciones (no se modif...,150000,43
3,categorías ordinales y nominales,150000,43
4,guardado en derivados,150000,43
